# 募集用途分析 重构

本项目用于分析债券募集用途文本，对比转型城投与普通城投的募集用途差异。

---

## 1. 项目概述

### 1.1 功能描述

**核心功能：**
1. 从数据库获取转型城投清单和募集用途数据
2. 使用jieba分词和关键词字典分析文本
3. 进行TF-IDF向量化与K-Means聚类分析
4. 生成各类可视化图表（词云、热力图、雷达图、饼图等）

### 1.2 数据源

- `yq.城投平台市场化经营主体` - 城投平台市场化经营主体清单
- `yq.城投平台退出` - 城投平台退出记录
- `yq.城投募集资金用途` - 城投债募集用途数据
- `yq.产业债募集资金用途` - 产业债募集用途数据

### 1.3 输出结果

- 关键词频率分析结果（JSON）
- 各类可视化图表（PNG）

## 2. 环境配置

### 2.1 导入依赖

In [ ]:
# 标准库
import os
import re
import json
import warnings
from datetime import datetime

# 数据处理
import numpy as np
import pandas as pd

# 数据库
import pymysql
from sqlalchemy import create_engine

# 文本分析
import jieba
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# 可视化
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from wordcloud import WordCloud
from matplotlib.font_manager import FontProperties

# 配置
from config import DB_CONFIG, KEYWORDS_DICT, OUTPUT_DIR

# 忽略警告
warnings.filterwarnings('ignore')

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

print("依赖导入完成")

### 2.2 配置参数

关键词字典定义了14个类别的关键词，用于分析募集用途文本。

In [ ]:
# 显示关键词字典
print("关键词字典配置:")
print("=" * 50)
for category, keywords in KEYWORDS_DICT.items():
    print(f"{category}: {', '.join(keywords)}")
print("=" * 50)
print(f"共 {len(KEYWORDS_DICT)} 个类别")

---

## 3. 数据获取

### 3.1 数据库连接

In [ ]:
def connect_db():
    """连接数据库"""
    print("正在连接数据库...")
    try:
        conn = pymysql.connect(
            host=DB_CONFIG['host'],
            user=DB_CONFIG['user'],
            password=DB_CONFIG['password'],
            database=DB_CONFIG['database'],
            port=DB_CONFIG['port'],
            charset='utf8'
        )
        print("数据库连接成功")
        return conn
    except Exception as e:
        print(f"数据库连接失败: {str(e)}")
        raise

# 测试连接
conn = connect_db()
print("连接测试成功")

### 3.2 转型城投清单查询

In [ ]:
def get_transform_companies(conn):
    """
    获取转型城投清单
    
    逻辑:
    - 从城投平台市场化经营主体表获取市场化经营的城投
    - 从城投平台退出表获取退出的城投（披露日期>=2023-10-26）
    - 使用UNION去重合并
    """
    query = """
    SELECT DISTINCT 公司名称 
    FROM yq.城投平台市场化经营主体
    UNION
    SELECT DISTINCT 公司名称 
    FROM yq.城投平台退出
    WHERE 披露日期 >= '2023-10-26'
    """
    print(f"\n执行查询: {query}")
    df = pd.read_sql(query, conn)
    companies = df['公司名称'].tolist()
    print(f"\n获取到 {len(companies)} 家转型城投")
    return companies

# 获取转型城投清单
transform_companies = get_transform_companies(conn)
print(f"\n转型城投样例（前5家）:")
for i, company in enumerate(transform_companies[:5], 1):
    print(f"  {i}. {company}")

### 3.3 募集用途数据获取

In [ ]:
def get_usage_samples(conn, time_period='1y'):
    """
    获取募集用途样本数据
    
    参数:
        conn: 数据库连接
        time_period: 时间范围 ('1y'=近1年, '3m'=近3个月)
    
    返回:
        (转型城投样本, 普通城投样本)
    """
    cursor = conn.cursor()
    
    # 时间条件
    if time_period == '1y':
        time_condition = "DATE_SUB(CURDATE(), INTERVAL 1 YEAR)"
        period_name = "近1年"
    else:
        time_condition = "DATE_SUB(CURDATE(), INTERVAL 3 MONTH)"
        period_name = "近3个月"
    
    # 获取转型城投样本
    print(f"执行转型城投查询({period_name})...")
    query_transform = f"""
    SELECT `原文`, `发行人`, `公告日期`, `发行规模(亿)`, `借新还旧(亿)`, 
           `偿还有息债务(亿)`, `补充流动资金(亿)`, `项目建设(亿)`, `其他(亿)`
    FROM 城投募集资金用途 
    WHERE 发行人 IN (
        SELECT DISTINCT 公司名称 FROM yq.城投平台市场化经营主体
        UNION
        SELECT DISTINCT 公司名称 FROM yq.城投平台退出 WHERE 披露日期 >= '2023-10-26'
    )
    AND 公告日期 >= {time_condition}
    UNION
    SELECT `原文`, `发行人`, `公告日期`, `发行规模(亿)`, `借新还旧(亿)`, 
           `偿还有息债务(亿)`, `补充流动资金(亿)`, `项目建设(亿)`, `其他(亿)`
    FROM 产业债募集资金用途
    WHERE 发行人 IN (
        SELECT DISTINCT 公司名称 FROM yq.城投平台市场化经营主体
        UNION
        SELECT DISTINCT 公司名称 FROM yq.城投平台退出 WHERE 披露日期 >= '2023-10-26'
    )
    AND 公告日期 >= {time_condition}
    """
    cursor.execute(query_transform)
    transform_samples = cursor.fetchall()
    print(f"获取到转型城投样本({period_name}): {len(transform_samples)}条")
    
    # 获取普通城投样本
    print(f"执行普通城投查询({period_name})...")
    query_normal = f"""
    SELECT `原文`, `发行人`, `公告日期`, `发行规模(亿)`, `借新还旧(亿)`, 
           `偿还有息债务(亿)`, `补充流动资金(亿)`, `项目建设(亿)`, `其他(亿)`
    FROM 城投募集资金用途 
    WHERE 发行人 NOT IN (
        SELECT DISTINCT 公司名称 FROM yq.城投平台市场化经营主体
        UNION
        SELECT DISTINCT 公司名称 FROM yq.城投平台退出 WHERE 披露日期 >= '2023-10-26'
    )
    AND 公告日期 >= {time_condition}
    """
    cursor.execute(query_normal)
    normal_samples = cursor.fetchall()
    print(f"获取到普通城投样本({period_name}): {len(normal_samples)}条")
    
    cursor.close()
    return transform_samples, normal_samples

# 获取样本数据
print("=" * 50)
print("获取近1年样本数据")
print("=" * 50)
transform_samples_1y, normal_samples_1y = get_usage_samples(conn, '1y')

print("\n" + "=" * 50)
print("获取近3个月样本数据")
print("=" * 50)
transform_samples_3m, normal_samples_3m = get_usage_samples(conn, '3m')

---

## 4. 数据处理

### 4.1 文本预处理

In [ ]:
def preprocess_text(text):
    """
    文本预处理
    
    处理逻辑:
    1. 清理空白字符
    2. 保留中文、英文、数字、空格
    3. 移除独立数字（如"123"），保留带单位数字（如"123亿"）
    4. 过滤过短文本（<10字符）
    """
    if not isinstance(text, str):
        return ""
    
    # 移除多余空白字符
    text = ' '.join(text.split())
    
    # 移除特殊字符，保留中文、英文、数字
    text = re.sub(r'[^\u4e00-\u9fa5a-zA-Z0-9\s]', ' ', text)
    
    # 移除独立数字，但保留带单位的数字
    text = re.sub(r'\b\d+\b(?!\s*[亿万元])', '', text)
    
    # 移除过短的文本
    if len(text.strip()) < 10:
        return ""
        
    return text.strip()

# 测试文本预处理
test_text = "本期债券募集资金用途为  123.45亿元用于偿还债务，50亿元用于补充流动资金。"
print(f"原始文本: {test_text}")
print(f"预处理后: {preprocess_text(test_text)}")

### 4.2 分词处理

In [ ]:
def tokenize_text(text):
    """
    使用jieba进行分词
    """
    if not isinstance(text, str) or not text.strip():
        return []
    return jieba.lcut(text)

# 测试分词
test_text = "本期债券募集资金用于偿还债务和补充流动资金"
words = tokenize_text(test_text)
print(f"原文: {test_text}")
print(f"分词结果: {' / '.join(words)}")

---

## 5. 核心逻辑

### 5.1 关键词分析

In [ ]:
def analyze_keywords(texts, keywords_dict):
    """
    分析文本中的关键词频率
    
    参数:
        texts: 文本列表
        keywords_dict: 关键词字典
    
    返回:
        各类别的关键词统计结果
    """
    total_words = 0
    keyword_counts = {
        category: {word: 0 for word in words} 
        for category, words in keywords_dict.items()
    }
    
    for text in texts:
        if not isinstance(text, str) or not text.strip():
            continue
        
        # 分词
        words = jieba.lcut(text)
        total_words += len(words)
        
        # 统计关键词
        for category, keywords in keywords_dict.items():
            for word in keywords:
                count = sum(1 for w in words if w == word)
                if count > 0:
                    keyword_counts[category][word] += count
    
    # 打印分析结果
    print(f"\n总词数: {total_words}")
    print("\n关键词频率分析结果:")
    for category, counts in keyword_counts.items():
        category_total = sum(counts.values())
        if category_total > 0:
            print(f"\n{category}:")
            for word, count in sorted(counts.items(), key=lambda x: x[1], reverse=True):
                if count > 0:
                    percentage = count / total_words * 100 if total_words > 0 else 0
                    print(f"  - {word}: {count}次 ({percentage:.2f}%)")
    
    return keyword_counts

### 5.2 聚类分析

In [ ]:
def perform_clustering(texts, n_clusters=5):
    """
    对文本进行聚类分析
    
    流程:
    1. 分词处理
    2. TF-IDF向量化（max_features=1000）
    3. K-Means聚类（n_clusters=5, random_state=42）
    4. 提取每个簇的top 10关键词
    
    参数:
        texts: 文本列表
        n_clusters: 聚类数量
    
    返回:
        (聚类标签, 簇关键词)
    """
    if not texts:
        print("输入文本列表为空")
        return None, None
    
    # 文本预处理
    processed_texts = []
    for text in texts:
        if not isinstance(text, str):
            text = str(text) if text is not None else ""
        
        text = text.strip()
        if not text:
            continue
        
        # 分词
        words = jieba.lcut(text)
        words = [w.strip() for w in words if w.strip() and len(w.strip()) > 1]
        if not words:
            continue
        
        processed_text = " ".join(words)
        if processed_text.strip():
            processed_texts.append(processed_text)
    
    if not processed_texts:
        print("预处理后没有有效的文本数据")
        return None, None
    
    print(f"有效文本数量: {len(processed_texts)}")
    
    try:
        # TF-IDF向量化
        vectorizer = TfidfVectorizer(
            max_features=1000,
            min_df=2,
            max_df=0.95,
            token_pattern=r"(?u)\b\w+\b"
        )
        X = vectorizer.fit_transform(processed_texts)
        print(f"向量化后的文本维度: {X.shape}")
        
        # 调整聚类数
        n_clusters = min(n_clusters, len(processed_texts))
        if n_clusters < 2:
            print("样本数量太少，无法进行聚类分析")
            return None, None
        
        # K-Means聚类
        kmeans = KMeans(
            n_clusters=n_clusters,
            random_state=42,
            n_init=10,
            max_iter=300,
            algorithm='elkan'
        )
        clusters = kmeans.fit_predict(X)
        print(f"聚类完成，共{n_clusters}个簇")
        
        # 提取簇关键词
        feature_names = vectorizer.get_feature_names_out()
        cluster_keywords = {}
        
        for i in range(n_clusters):
            cluster_docs = X[clusters == i]
            if cluster_docs.shape[0] == 0:
                continue
            
            centroid = cluster_docs.mean(axis=0).A1
            sorted_indices = centroid.argsort()[::-1][:10]
            keywords = [feature_names[j] for j in sorted_indices]
            cluster_keywords[i] = keywords
            
            print(f"\n簇 {i} (包含{cluster_docs.shape[0]}个文档)的关键词:")
            print(", ".join(keywords))
        
        return clusters, cluster_keywords
        
    except Exception as e:
        print(f"聚类过程出错: {str(e)}")
        return None, None

### 5.3 时间模式分析

In [ ]:
def analyze_time_patterns(samples, sample_type=""):
    """
    时间模式分析
    
    分析维度:
    - 月度发行数量
    - 月度发行规模
    """
    print(f"\n{sample_type}时间分析:")
    
    # 转换为DataFrame
    df = pd.DataFrame(samples, columns=[
        '原文', '发行人', '公告日期', '发行规模(亿)', '借新还旧(亿)', 
        '偿还有息债务(亿)', '补充流动资金(亿)', '项目建设(亿)', '其他(亿)'
    ])
    
    # 转换公告日期为datetime
    df['公告日期'] = pd.to_datetime(df['公告日期'])
    
    # 按月份统计发行数量
    monthly_counts = df.groupby(df['公告日期'].dt.to_period('M')).size()
    print("\n月度发行数量:")
    for month, count in monthly_counts.items():
        print(f"  {month}: {count}笔")
    
    # 按月份统计发行规模
    df['发行规模(亿)'] = pd.to_numeric(df['发行规模(亿)'], errors='coerce')
    monthly_amounts = df.groupby(df['公告日期'].dt.to_period('M'))['发行规模(亿)'].sum()
    print("\n月度发行规模(亿元):")
    for month, amount in monthly_amounts.items():
        print(f"  {month}: {amount:.2f}亿")
    
    return df

---

## 6. 执行与测试

### 6.1 执行主流程

In [ ]:
# 分析转型城投样本(近1年)
print("=" * 60)
print("=== 转型城投样本分析(近1年) ===")
print("=" * 60)

if transform_samples_1y:
    # 提取文本
    transform_texts_1y = [sample[0] for sample in transform_samples_1y if sample[0]]
    
    # 关键词分析
    transform_counts_1y = analyze_keywords(transform_texts_1y, KEYWORDS_DICT)
    
    # 聚类分析
    transform_clusters_1y, transform_keywords_1y = perform_clustering(transform_texts_1y)
    
    # 时间分析
    transform_df_1y = analyze_time_patterns(transform_samples_1y, "转型城投(近1年)")
else:
    print("未获取到转型城投样本数据")

In [ ]:
# 分析普通城投样本(近1年)
print("=" * 60)
print("=== 普通城投样本分析(近1年) ===")
print("=" * 60)

if normal_samples_1y:
    # 提取文本
    normal_texts_1y = [sample[0] for sample in normal_samples_1y if sample[0]]
    
    # 关键词分析
    normal_counts_1y = analyze_keywords(normal_texts_1y, KEYWORDS_DICT)
    
    # 聚类分析
    normal_clusters_1y, normal_keywords_1y = perform_clustering(normal_texts_1y)
    
    # 时间分析
    normal_df_1y = analyze_time_patterns(normal_samples_1y, "普通城投(近1年)")
else:
    print("未获取到普通城投样本数据")

### 6.2 可视化展示

In [ ]:
def ensure_output_dir():
    """确保输出目录存在"""
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)
        print(f"创建输出目录: {OUTPUT_DIR}")

def add_data_source(fig):
    """添加数据来源说明"""
    fig.text(0.99, 0.01, '数据来源：弘则研究，企业发行人募集说明书', 
             ha='right', va='bottom', fontsize=8, style='italic')

# 确保输出目录存在
ensure_output_dir()
print(f"输出目录: {OUTPUT_DIR}")

In [ ]:
def plot_keyword_heatmap(keyword_counts, sample_type=""):
    """绘制关键词热力图"""
    fig = plt.figure(figsize=(15, 10))
    
    # 准备数据
    categories = list(keyword_counts.keys())
    keywords = []
    frequencies = []
    cats = []
    
    for category in categories:
        for keyword, count in keyword_counts[category].items():
            keywords.append(keyword)
            frequencies.append(count)
            cats.append(category)
    
    # 创建DataFrame
    df = pd.DataFrame({
        'Category': cats,
        'Keyword': keywords,
        'Frequency': frequencies
    })
    
    # 转换为透视表
    pivot_table = df.pivot(index='Category', columns='Keyword', values='Frequency')
    
    # 绘制热力图
    sns.heatmap(pivot_table, cmap='YlOrRd', annot=True, fmt='.0f')
    plt.title(f'{sample_type}关键词频率热力图')
    plt.tight_layout()
    
    # 添加数据来源
    add_data_source(fig)
    
    plt.savefig(os.path.join(OUTPUT_DIR, f'{sample_type}_keyword_heatmap.png'), dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

# 生成转型城投热力图
if transform_samples_1y:
    plot_keyword_heatmap(transform_counts_1y, "转型城投(近1年)")

In [ ]:
def plot_wordcloud(texts, sample_type=""):
    """生成词云图"""
    fig = plt.figure(figsize=(15, 10))
    
    # 合并所有文本
    text = ' '.join([str(t) for t in texts if isinstance(t, str)])
    
    # 定义停用词列表
    stopwords = {'募集', '资金', '全部', '用于', '本期', '债券', '发行', '费用', '亿元', 
                '本金', '发', '本', '的', '了', '和', '与', '及', '将'}
    
    # 获取所有关键词
    all_keywords = set()
    for keywords in KEYWORDS_DICT.values():
        all_keywords.update(keywords)
    
    # 分词并只保留关键词列表中的词
    words = jieba.lcut(text)
    keyword_freq = {}
    for word in words:
        if word in all_keywords and word not in stopwords:
            keyword_freq[word] = keyword_freq.get(word, 0) + 1
    
    if not keyword_freq:
        print("没有找到关键词，无法生成词云图")
        return
    
    # 生成词云
    wordcloud = WordCloud(
        width=1200, height=800,
        background_color='white',
        font_path='C:/Windows/Fonts/simhei.ttf',
        min_font_size=10,
        max_font_size=100,
        max_words=100,
        collocations=False
    ).generate_from_frequencies(keyword_freq)
    
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title(f'{sample_type}关键词词云图')
    plt.tight_layout()
    
    add_data_source(fig)
    
    plt.savefig(os.path.join(OUTPUT_DIR, f'{sample_type}_wordcloud.png'), dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

# 生成转型城投词云
if transform_samples_1y:
    plot_wordcloud(transform_texts_1y, "转型城投(近1年)")

In [ ]:
def plot_radar_chart(keyword_counts, sample_type=""):
    """绘制雷达图"""
    fig = plt.figure(figsize=(10, 10))
    
    # 准备数据
    categories = list(keyword_counts.keys())
    values = [sum(keyword_counts[cat].values()) for cat in categories]
    
    # 计算角度
    angles = np.linspace(0, 2*np.pi, len(categories), endpoint=False)
    
    # 闭合图形
    values = np.concatenate((values, [values[0]]))
    angles = np.concatenate((angles, [angles[0]]))
    
    # 绘制雷达图
    ax = plt.subplot(111, projection='polar')
    ax.plot(angles, values)
    ax.fill(angles, values, alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories)
    
    plt.title(f'{sample_type}维度分布雷达图')
    plt.tight_layout()
    
    add_data_source(fig)
    
    plt.savefig(os.path.join(OUTPUT_DIR, f'{sample_type}_radar_chart.png'), dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

# 生成转型城投雷达图
if transform_samples_1y:
    plot_radar_chart(transform_counts_1y, "转型城投(近1年)")

In [ ]:
def plot_usage_pie(df, sample_type=""):
    """绘制资金用途饼图"""
    fig = plt.figure(figsize=(12, 8))
    
    # 准备数据
    usage_cols = ['借新还旧(亿)', '偿还有息债务(亿)', '补充流动资金(亿)', '项目建设(亿)', '其他(亿)']
    values = [df[col].astype(float).sum() for col in usage_cols]
    total = sum(values)
    percentages = [v/total*100 for v in values]
    
    # 绘制饼图
    plt.pie(percentages, labels=usage_cols, autopct='%1.1f%%')
    plt.title(f'{sample_type}资金用途分布')
    plt.tight_layout()
    
    add_data_source(fig)
    
    plt.savefig(os.path.join(OUTPUT_DIR, f'{sample_type}_usage_pie.png'), dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

# 生成转型城投饼图
if transform_samples_1y:
    plot_usage_pie(transform_df_1y, "转型城投(近1年)")

In [ ]:
def plot_monthly_trend(df, sample_type=""):
    """绘制月度趋势图"""
    fig = plt.figure(figsize=(15, 8))
    
    # 转换日期
    df['公告日期'] = pd.to_datetime(df['公告日期'])
    monthly_data = df.groupby(df['公告日期'].dt.to_period('M'))['发行规模(亿)'].sum()
    
    # 绘制趋势线
    plt.plot(range(len(monthly_data)), monthly_data.values, marker='o')
    plt.xticks(range(len(monthly_data))[::3], [str(m) for m in monthly_data.index[::3]], rotation=45)
    plt.title(f'{sample_type}月度发行规模趋势')
    plt.ylabel('发行规模(亿元)')
    plt.grid(True)
    plt.tight_layout()
    
    add_data_source(fig)
    
    plt.savefig(os.path.join(OUTPUT_DIR, f'{sample_type}_monthly_trend.png'), dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

# 生成转型城投月度趋势图
if transform_samples_1y:
    plot_monthly_trend(transform_df_1y.copy(), "转型城投(近1年)")

In [ ]:
def plot_keyword_comparison(counts1, counts2, label1="组1", label2="组2", time_period=""):
    """绘制关键词对比图"""
    # 计算每个样本的总词频
    def get_total_freq(keyword_counts):
        total = 0
        for category in keyword_counts.values():
            total += sum(category.values())
        return total
    
    total_freq1 = get_total_freq(counts1)
    total_freq2 = get_total_freq(counts2)
    
    # 对每个类别生成对比图（仅展示部分）
    categories_to_show = ['债务管理', '转型相关', '产业投资']
    
    for category in categories_to_show:
        if category not in counts1:
            continue
            
        fig = plt.figure(figsize=(12, 6))
        
        # 获取该类别下的所有关键词
        keywords = list(set(counts1[category].keys()) | set(counts2[category].keys()))
        
        # 计算频率占比
        values1 = [(counts1[category].get(k, 0) / total_freq1 * 100) if total_freq1 > 0 else 0 for k in keywords]
        values2 = [(counts2[category].get(k, 0) / total_freq2 * 100) if total_freq2 > 0 else 0 for k in keywords]
        
        # 设置条形图
        x = np.arange(len(keywords))
        width = 0.35
        
        plt.bar(x - width/2, values1, width, label=label1)
        plt.bar(x + width/2, values2, width, label=label2)
        
        plt.xlabel('关键词')
        plt.ylabel('频率占比(%)')
        plt.title(f'{category}关键词频率对比')
        plt.xticks(x, keywords, rotation=45, ha='right')
        plt.legend()
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.tight_layout()
        
        add_data_source(fig)
        
        plt.savefig(os.path.join(OUTPUT_DIR, f'{category}_comparison_{time_period}.png'), dpi=300, bbox_inches='tight')
        plt.show()
        plt.close()

# 生成对比图（近1年）
if transform_samples_1y and normal_samples_1y:
    plot_keyword_comparison(transform_counts_1y, normal_counts_1y, 
                          "转型城投(近1年)", "普通城投(近1年)", "近1年")

---

## 7. 工具函数（可复用）

In [ ]:
def save_analysis_results(results, filename=None):
    """
    保存分析结果到JSON文件
    
    参数:
        results: 分析结果字典
        filename: 输出文件名（可选）
    """
    # 生成文件名
    if filename is None:
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        filename = f'analysis_results_{timestamp}.json'
    
    # 确保输出目录存在
    ensure_output_dir()
    
    # 保存结果
    output_path = os.path.join(OUTPUT_DIR, filename)
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    
    print(f"\n分析结果已保存到: {output_path}")
    return output_path

# 保存分析结果
if transform_samples_1y and normal_samples_1y:
    results = {
        'timestamp': datetime.now().isoformat(),
        'transform_1y': {
            'sample_count': len(transform_samples_1y),
            'keyword_counts': transform_counts_1y
        },
        'normal_1y': {
            'sample_count': len(normal_samples_1y),
            'keyword_counts': normal_counts_1y
        }
    }
    save_analysis_results(results)

In [ ]:
# 关闭数据库连接
if 'conn' in locals() and conn.open:
    conn.close()
    print("数据库连接已关闭")

print("\n" + "=" * 60)
print("分析完成！")
print("=" * 60)